# CI failure prediction on GitHub Actions - feature selection

Chooses the features the tree is trained on, on the **pooled** dataset: the enabled targets of `config/targets.json`, analysed as one CI workflow. Selection uses **only the dev segment** (the first 20% of the pooled runs), which comes before every CV test fold, so no test label influences the choice. The resulting list is **frozen** and used unchanged in the CV, the tuning and the final model.

Pipeline (same as project 1): clean -> relevance -> redundancy pruning -> stability over block subsamples -> frozen list, with an embedded random-forest check and a drift check on the CV trains.

**Main target: T2.** T1 is selected too, for the comparison with the Parrot baseline.

**Excluded before selection:** `EXCLUDE` in the setup cell, decided from the pooled EDA (`eda.ipynb`): features whose dev-segment signal does not hold later, or whose values split time periods instead of changes. It starts empty.

In [ ]:
from dataclasses import replace

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from sklearn.metrics import matthews_corrcoef

from acise_p2_cidt.analysis import features as fs
from acise_p2_cidt.collect.targets import load_targets, slug
from acise_p2_cidt.data.load import PREV_LABEL_COL, Dataset, load, pool_name, subset
from acise_p2_cidt.training.cv import folds
from acise_p2_cidt.viz import plots

pd.set_option("display.max_rows", 100)

SOURCES = [slug(c.target) for c in load_targets()]
DATASET = pool_name(SOURCES)
SEED = 42
MIN_FREQ = 0.6  # minimum share of subsamples a feature must be selected in
RHO = 0.8  # redundancy threshold (|Spearman|)
N_MAX = 20  # top-k per subsample, and cap on the frozen list
EXCLUDE: tuple[str, ...] = ()  # set from the pooled EDA
E1_MIN_FAIL = 50  # T2 failures a source needs for the per-source experiments


def dev(ds: Dataset) -> tuple[pd.DataFrame, pd.Series]:
    return ds.X[ds.dev_mask].reset_index(drop=True), ds.y[ds.dev_mask].reset_index(drop=True)


t1, t2 = load(SOURCES, "T1"), load(SOURCES, "T2")
X1, y1 = dev(t1)
X2, y2 = dev(t2)

display(
    Markdown(f"**Dev segment of the pooled dataset `{DATASET}`** ({len(SOURCES)} sources): rows, failures, "
             "failure rate and candidate features, T1 and T2."),
    pd.DataFrame(
        {
            tag: {
                "dev_rows": len(y),
                "dev_failed": int(y.sum()),
                "fail_%": round(y.mean() * 100, 2),
                "features": X.shape[1],
            }
            for tag, X, y in (("T1", X1, y1), ("T2", X2, y2))
        }
    ),
    Markdown("**Dev rows per source**, T1 and T2: a source that starts late has few or no dev rows."),
    pd.DataFrame(
        {ds.meta["target"]: ds.source[ds.dev_mask].value_counts() for ds in (t1, t2)}
    ).reindex(SOURCES).fillna(0).astype(int),
)

## 1. Clean (T2)

Filters, fitted on the pooled dev rows:

- **Excluded**: `EXCLUDE` (setup cell).
- **Missing > 30%**: dropped; the rest is median-imputed.
- **Near-constant**: `VarianceThreshold(1e-3)` on **percentile ranks**. Ranks keep a truly constant column at variance 0, while one huge outlier can no longer push a count column below the threshold.
- **Rare two-valued**: a column with two values whose minority value is on < 1% of the rows (0/1 flags, and `n_parents`, which is 1 or 2).

In [ ]:
X2e = X2.drop(columns=list(EXCLUDE))
Xc = fs.clean(X2e)


def reason(c: str) -> str:
    if c in EXCLUDE:
        return "excluded"
    # as `clean`: a lineage's first run has no history, so it is not counted as missing
    if X2.loc[X2["lineage_first_run"] != 1, c].isna().mean() > 0.3:
        return "missing"
    return "rare two-valued" if X2[c].nunique() == 2 else "near-constant"


dropped = sorted(set(X2.columns) - set(Xc.columns))
display(
    Markdown(f"**Features removed by cleaning** (T2, pooled dev rows): {X2.shape[1]} -> {Xc.shape[1]} features, "
             f"{len(dropped)} removed, with the reason"),
    pd.Series({c: reason(c) for c in dropped}, name="reason").to_frame(),
)

## 2. Relevance (T2)

The statistic depends on the feature type:

1. Numeric: ANOVA F, |Kendall tau|, mutual information (MI).
2. Binary: chi-square, MI (= information gain).

Counts and churn are heavily skewed, so rank- and information-based measures are used, not Pearson. `score` is the mean **percentile** rank over the statistics that apply, so numeric and binary features are comparable. MI also catches threshold effects that the monotone statistics miss, which is what a tree split uses.

In [ ]:
scores = fs.relevance(Xc, y2, seed=SEED)
display(
    Markdown("**Relevance per statistic** (T2, pooled dev rows, top 25); `score` is the mean percentile rank"),
    scores.head(25).round(4),
)

ranks = scores.drop(columns="score").dropna(axis=1, how="all").rank(pct=True)  # no binary left: chi2 is empty
fig = plots.scores_bar(ranks, top=25)
plots.save(fig.update_layout(title="Relevance percentile rank per statistic (T2, pooled dev rows, top 25)"), "fs_relevance_T2")

## 3. Redundancy (T2)

mRMR-style walk down the relevance ranking: a feature is kept only if its |Spearman rho| with **every** feature kept so far is at most 0.8. Each family of near-duplicates keeps its most relevant member, its **representative**.

In [ ]:
ranking = scores.index.tolist()
kept = fs.prune_redundant(Xc, ranking, rho=RHO)
print(f"{len(ranking)} -> {len(kept)} features after redundancy pruning (rho={RHO})")

rep = fs._family_rep(Xc, kept, rho=RHO)
fam = pd.Series(rep, name="representative").to_frame()
fam = fam[fam.index != fam["representative"]]
fam["abs_spearman"] = [abs(Xc[f].corr(Xc[r], method="spearman")) for f, r in fam["representative"].items()]
display(
    Markdown(f"**Near-duplicate families** (T2, pooled dev rows, |Spearman| > {RHO}): each pruned feature and the "
             "representative kept in its place"),
    fam.sort_values(["representative", "abs_spearman"], ascending=[True, False]).round(4),
)

Clean, relevance and redundancy are repeated on 20 **block subsamples** of the dev rows: 80% of contiguous blocks (length sqrt3(n)) drawn without replacement and kept in time order. A feature is selected in a subsample if it is in the top `N_MAX = 20` after pruning; a pick of a near-duplicate is credited to its family representative.

The **frozen list** is the pruned features with frequency >= `MIN_FREQ`, in relevance order, capped at 20.

In [ ]:
# sel_dev keeps the dev-only list, so the drift cell below can be re-run safely
sel = sel_dev = fs.select_features(t2, n_max=N_MAX, min_freq=MIN_FREQ, rho=RHO, seed=SEED, exclude=EXCLUDE)
assert sel.scores.index.tolist() == ranking, "select_features must see the same cleaned dev rows"
display(
    Markdown(f"**Frozen list from the dev rows** (T2, N = {sel.size}): stability over the subsamples and relevance score"),
    pd.DataFrame(
        {"stability": sel.stability[sel.features], "score": sel.scores.loc[sel.features, "score"]}
    ).round(4),
)

In [ ]:
fig = plots.stability_bar(sel.stability[sel.stability > 0])
fig.add_hline(y=MIN_FREQ, line_dash="dash", annotation_text=f"min_freq={MIN_FREQ}", annotation_position="bottom right")
plots.save(fig.update_layout(title="Feature selection stability over subsamples (T2, pooled dev rows)"), "fs_stability_T2")

How the list changes with the frequency threshold. A list that changes a lot between neighbouring thresholds would mean the choice of `MIN_FREQ` drives the result.

In [ ]:
THRESHOLDS = [0.5, 0.6, 0.75, 0.9, 0.95]


def frozen(t: float) -> list[str]:
    return [f for f in kept if sel.stability.get(f, 0) >= t][:N_MAX]


assert frozen(MIN_FREQ) == sel.features, "frozen list must match select_features"
display(
    Markdown("**Frozen list per stability threshold** (T2): list size N and the features"),
    pd.DataFrame(
        [{"min_freq": t, "N": len(frozen(t)), "features": ", ".join(frozen(t))} for t in THRESHOLDS]
    ).set_index("min_freq"),
)

## 5. Embedded check (T2)

A random forest (300 trees, `class_weight="balanced"`) is fitted on the first 70% of the dev rows; permutation importance (ROC AUC, 10 repeats) is measured on the chronological last 30%. Features in the forest's top 5 that the filters dropped are listed as **disagreements**. The check never changes the frozen list.

In [ ]:
imp = sel.importance.head(20).rename("importance").rename_axis("feature").reset_index()
imp["selected"] = imp["feature"].isin(sel.features)

display(
    Markdown("**Disagreements** (T2): features in the forest's top 5 that the filters dropped, with their importance, "
             "stability and family representative"),
    pd.DataFrame(
        {
            "importance": sel.importance[sel.disagreements],
            "stability": sel.stability.reindex(sel.disagreements),
            "family": [rep.get(f, f) for f in sel.disagreements],
        }
    ).round(4),
)
plots.save(
    px.bar(imp, x="feature", y="importance", color="selected")
    .update_layout(title="Permutation importance (T2, pooled dev rows, top 20)"),
    "fs_importance_T2",
)

## 6. Drift (T2)

Selection is re-run on each CV fold's train rows (dev + the expanding window, pooled). A frozen feature counts as **kept** in a fold if it, or a near-duplicate, is in that fold's top 20.

As in project 1, a frozen feature lost in **3 or more of the 5 folds** is removed. **Caveat:** the fold trains contain labels after the dev segment, including rows that are test rows of earlier folds, so this step uses later labels to change the list. Removing features can only shrink the list, never add a feature that the dev rows did not pick, but the CV results of the tree must be read with this in mind.

In [ ]:
cv_folds = folds(t2)
print("fold train ends:", [str(t2.order.iloc[f.train[-1]].date()) for f in cv_folds])

t2e = replace(t2, X=t2.X.drop(columns=list(EXCLUDE)))
dr = fs.drift(t2e, [f.train for f in cv_folds], sel_dev.features, n_max=N_MAX, rho=RHO, seed=SEED)

display(
    Markdown("**Share of the 5 CV trains where each frozen feature is kept** (T2, pooled)"),
    dr.mean().round(3).rename("share kept").to_frame().T,
)

DRIFT_DROP = dr.index[(~dr).sum(axis=1) >= 3].tolist()
sel = replace(sel_dev, features=[f for f in sel_dev.features if f not in DRIFT_DROP], meta=sel_dev.meta | {"dropped_due_to_drift": DRIFT_DROP})
print(f"dropped due to drift ({len(DRIFT_DROP)}):", DRIFT_DROP)
print("N =", sel.size, "|", sel.features)

plots.save(plots.drift_heatmap(dr).update_layout(title="Feature selection drift over the CV trains (T2, pooled)"), "fs_drift_T2")

### 6.1 Per-source lists (E1)

The per-source experiment trains one tree per source on its own data, so each source gets its own list: the same selection, on that source's rows of the pooled dev segment. Sources with fewer than `E1_MIN_FAIL` T2 failures in total, or fewer than 10 failures among their dev rows, get no list. The Jaccard index with the pooled list shows how far the pool's choice is from each source's own choice. Drift filtering is not repeated per source.

In [ ]:
def jaccard(a: list[str], b: list[str]) -> float:
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else np.nan


src2 = t2.source.to_numpy()
dev_fail = t2.y[t2.dev_mask].groupby(t2.source[t2.dev_mask]).sum()
E1_SOURCES = [s for s in SOURCES if t2.meta["per_source"][s]["n_failed"] >= E1_MIN_FAIL and dev_fail.get(s, 0) >= 10]
sel_src = {
    s: fs.select_features(subset(t2, src2 == s, dataset=s), n_max=N_MAX, min_freq=MIN_FREQ, rho=RHO, seed=SEED, exclude=EXCLUDE)
    for s in E1_SOURCES
}
display(
    Markdown(f"**Per-source lists** (T2, each source's dev rows): size, dev rows and failures, Jaccard with the pooled "
             f"list; {len(SOURCES) - len(E1_SOURCES)} source(s) without a list"),
    pd.DataFrame(
        {
            s: {"N": x.size, "dev_rows": x.meta["dev_rows"], "dev_failed": x.meta["dev_failed"],
                "jaccard_vs_pool": round(jaccard(x.features, sel.features), 3)}
            for s, x in sel_src.items()
        }
    ).T,
    Markdown("**Per-source lists, side by side** (T2), in relevance order"),
    pd.DataFrame({s: pd.Series(x.features) for s, x in sel_src.items()}),
)

## 7. T1 and the Parrot input

On T1, `prev_failed` is the Parrot baseline's only input: the previous completed run **of the same lineage** (the same PR or branch) failed. In the pool every source keeps its own lineages, so Parrot never reads a run of another repository. `prev_failed_jobs` (failed jobs of that previous run) and `fail_streak` carry the same information (rho ~= 1 with `prev_failed`), so the comparison removes the whole **Parrot family**. `prev_failed` is missing on the first run of every lineage, so on a pool with many short PR lineages cleaning can remove it (> 30% missing); the table then shows no rank for it. T1 is a secondary target: its list is not drift-filtered.

In [ ]:
sel1 = fs.select_features(t1, n_max=N_MAX, min_freq=MIN_FREQ, rho=RHO, seed=SEED, exclude=EXCLUDE)
PARROT = (PREV_LABEL_COL, "prev_failed_jobs", "fail_streak")
sel1_np = fs.select_features(t1, n_max=N_MAX, min_freq=MIN_FREQ, rho=RHO, seed=SEED, exclude=EXCLUDE + PARROT)

display(
    Markdown("**Parrot family on T1** (pooled dev rows): relevance rank and selection stability of each member; "
             "no rank = removed by cleaning"),
    pd.DataFrame(
        {
            "rank": [sel1.scores.index.get_loc(c) + 1 if c in sel1.scores.index else None for c in PARROT],
            "stability": sel1.stability.reindex(list(PARROT)),
        },
        index=PARROT,
    ),
    Markdown("**Spearman correlation within the Parrot family** (T1, pooled dev rows)"),
    X1[list(PARROT)].corr(method="spearman").round(3),
)
print("Parrot MCC (T1 dev):", round(matthews_corrcoef(y1, X1[PREV_LABEL_COL].fillna(0)), 4))
display(
    Markdown("**Frozen lists side by side**: T2, T1, and T1 without the Parrot family"),
    pd.DataFrame(
        {
            "T2": pd.Series(sel.features),
            "T1": pd.Series(sel1.features),
            "T1 without Parrot family": pd.Series(sel1_np.features),
        }
    ),
)

In [ ]:
print(f"N: T2 {sel.size}, T1 {sel1.size}, T1 w/o Parrot family {sel1_np.size}")
print("Jaccard T1 vs T2:", round(jaccard(sel.features, sel1.features), 3))
print("Jaccard T1 vs T1 w/o Parrot family:", round(jaccard(sel1.features, sel1_np.features), 4))

## 8. Freeze

Writes `results/features/selected_features_<target>_<DATASET>.json`, where `DATASET` is the pool name (`pool-` and a hash of the sorted source slugs, so a changed target list never overwrites an earlier pool): the list, N, scores, stability, importance, disagreements, the selection parameters (including the exclusions and, for T2, the drift drops), and the source slugs. `tree.ipynb` reads it with `load_features`.

In [ ]:
for s in (sel, sel1, *sel_src.values()):
    print(s.save(), "| N =", s.size)

## Observations

_To be written after the run on the pooled data._